## Data processing

## 1. Libraries import

In [1]:
import pandas as pd
import numpy as np
import math
import re

## 2. Data import

In [ ]:
df = pd.read_csv("camptocamp_routes_full.csv")
df

,document_id,title,title_prefix,quality,activities,climbing_outdoor_type,global_rating,engagement_rating,risk_rating,equipment_rating,...,rock_types,orientations,configuration,elevation_min,elevation_max,height_diff_up,height_diff_down,height_diff_difficulties,durations,description
0,1948187,Triangle des Coutchets - Brouillard d'été,Pic des Coutchets,draft,rock_climbing,multi,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,A gauche de Aquim
1,1948176,Triangle des Coutchets - Soleil d'automne,Pic des Coutchets,medium,rock_climbing,multi,TD-,NaN,NaN,P2,...,calcaire,NE,face,NaN,NaN,NaN,NaN,150.0,1,Approche: monter au plateau de Bezou soit à pi...
2,1944609,Capricho,Aiguille de la Canalona,medium,rock_climbing,multi,D+,NaN,NaN,P2+,...,calcaire,S,pillar,1850.0,2540.0,690.0,NaN,170.0,1,"## Approche ##\n\nDepuis le Câble, on suit la ..."
3,1944282,Secteur Rocaïne - Cash cache,Les Douanes,fine,rock_climbing,single,NaN,NaN,NaN,NaN,...,calcaire,S,NaN,1560.0,1610.0,100.0,100.0,15.0,1,## Approche # 15min\nDu parking en bordure de ...
4,1944020,Face W - Heureux qui comme Ulysse,Pène Sarrière,fine,rock_climbing,multi,ED-,I,X2,P1,...,calcaire,W,face,1370.0,1830.0,NaN,NaN,195.0,1,## Approche # 45min\nSuivre l'approche de la [...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
16855,54116,L'avenue des chamois,Motteux Péteux,fine,rock_climbing,multi,AD+,II,NaN,P1+,...,calcaire,S,face,1633.0,2232.0,680.0,680.0,300.0,1,"## Approche # 1h15\nDe Mittelberg (1632 m), su..."
16856,53904,Voie Originale de l'Y,Miroir d'Argentine,medium,rock_climbing,multi,D-,II,NaN,P1+,...,NaN,N,face,1469.0,2325.0,850.0,892.0,380.0,1,"## Approche # (env. 1h)\nDu parking supérieur,..."
16857,53903,Papageno,Miroir d'Argentine,fine,rock_climbing,multi,D,II,NaN,P2,...,NaN,NW,face,1462.0,2325.0,800.0,887.0,400.0,1,"## Approche\nDu parking, traverser le torrent ..."
16858,53893,Voie Directe,Miroir d'Argentine,fine,rock_climbing,multi,D,II,NaN,P2,...,calcaire,NW,face,1462.0,2325.0,800.0,800.0,400.0,1,[img=1045456 right]Tracé de la voie[/img]\n\n#...


## 3. First data cleaning

In [ ]:
# Remove all bouldering or psicobloc routes because their descriptions are not rich enough for this project
df = df[df['climbing_outdoor_type'] != 'bloc'].reset_index(drop=True)
df = df[df['climbing_outdoor_type'] != 'psicobloc'].reset_index(drop=True)

# Remove rows where the route type is missing (NaN)
df = df.dropna(subset=["climbing_outdoor_type"])

# Remove all routes without a description
df = df.dropna(subset=['description']).reset_index(drop=True)

In [19]:
df

,document_id,title,title_prefix,quality,activities,climbing_outdoor_type,global_rating,engagement_rating,risk_rating,equipment_rating,...,rock_types,orientations,configuration,elevation_min,elevation_max,height_diff_up,height_diff_down,height_diff_difficulties,durations,description
0,1948187,Triangle des Coutchets - Brouillard d'été,Pic des Coutchets,draft,rock_climbing,multi,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,A gauche de Aquim
1,1948176,Triangle des Coutchets - Soleil d'automne,Pic des Coutchets,medium,rock_climbing,multi,TD-,NaN,NaN,P2,...,calcaire,NE,face,NaN,NaN,NaN,NaN,150.0,1,Approche: monter au plateau de Bezou soit à pi...
2,1944609,Capricho,Aiguille de la Canalona,medium,rock_climbing,multi,D+,NaN,NaN,P2+,...,calcaire,S,pillar,1850.0,2540.0,690.0,NaN,170.0,1,"## Approche ##\n\nDepuis le Câble, on suit la ..."
3,1944282,Secteur Rocaïne - Cash cache,Les Douanes,fine,rock_climbing,single,NaN,NaN,NaN,NaN,...,calcaire,S,NaN,1560.0,1610.0,100.0,100.0,15.0,1,## Approche # 15min\nDu parking en bordure de ...
4,1944020,Face W - Heureux qui comme Ulysse,Pène Sarrière,fine,rock_climbing,multi,ED-,I,X2,P1,...,calcaire,W,face,1370.0,1830.0,NaN,NaN,195.0,1,## Approche # 45min\nSuivre l'approche de la [...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
14245,54143,"Zygofolis (départ ""Directe"")",Miroir d'Argentine,medium,rock_climbing,multi,TD+,II,NaN,P1+,...,NaN,NW,NaN,1462.0,2325.0,850.0,NaN,450.0,1,## Voie\nL# | 5a+ | | De la [[routes/53893|Vo...
14246,54116,L'avenue des chamois,Motteux Péteux,fine,rock_climbing,multi,AD+,II,NaN,P1+,...,calcaire,S,face,1633.0,2232.0,680.0,680.0,300.0,1,"## Approche # 1h15\nDe Mittelberg (1632 m), su..."
14247,53904,Voie Originale de l'Y,Miroir d'Argentine,medium,rock_climbing,multi,D-,II,NaN,P1+,...,NaN,N,face,1469.0,2325.0,850.0,892.0,380.0,1,"## Approche # (env. 1h)\nDu parking supérieur,..."
14248,53903,Papageno,Miroir d'Argentine,fine,rock_climbing,multi,D,II,NaN,P2,...,NaN,NW,face,1462.0,2325.0,800.0,887.0,400.0,1,"## Approche\nDu parking, traverser le torrent ..."


## 4. Splitting routes into pitches

In [ ]:
# ================================================================
# 1. IDENTIFYING PITCHES
# ================================================================

motif_longueur = re.compile(
    r"(?mi)^[ \t]*"
    r"(?P<label>"
        r"L#(?:\d+(?:-\d+)?(?:bis|ter)?[A-Za-z']*)?"
        r"|L\d+(?:-\d+)?(?:bis|ter)?[A-Za-z']*"
        r"|P\d+(?:-\d+)?(?:bis|ter)?[A-Za-z']*"
    r")"
    r"(?!\d)"
    r"(?!~)"              
    r"[ \t]*"
)


# ================================================================
# 2. SECTIONS / HEADINGS
# ================================================================

# Any level-2 or level-3 Markdown heading acts as a boundary.
#
# Examples:
#
# ## Route
# ## Return
# ### Latay Slab
# ### Tuet Slab
# ### Small Pillar
# ### Bastion
# ### Pillar
#
# The heading itself must never be included in the description
# of a pitch.
# ================================================================

motif_titre_section = re.compile(
    r"(?mi)^[ \t]*"
    r"#{2,3}[ \t]*"
    r"[^\n]*"
    r"$"
)


# ================================================================
# 3. SECTIONS THAT ACTUALLY END THE ROUTE SECTION
# ================================================================

motif_fin_voie = re.compile(
    r"(?mi)^[ \t]*#{1,3}[ \t]*"
    r"(Descente|Retour|Approche|Itin[ée]raire|Acc[eè]s|Mat[ée]riel|Topo)"
    r"\b"
)


# ================================================================
# 4. HEIGHT DETECTION
# ================================================================

motif_hauteur = re.compile(
    r"^\s*"

    # --------------------------------------------------------
    # Optional Markdown italics: _25m_
    # --------------------------------------------------------
    r"_?"

    # --------------------------------------------------------
    # Optional approximation marker:
    #   ~30 m   <25m   >40 m   Env. 30 m   environ 30 m
    # --------------------------------------------------------
    r"(?P<approx>~|<|>|environ\b|env\.?)?"
    r"\s*"

    r"(?:"

        # Range: 30/40 m, 30-35m
        r"(?P<valeur_plage>"
            r"\d+(?:[.,]\d+)?"
            r"(?:/|-)"
            r"\d+(?:[.,]\d+)?"
        r")"

        r"|"

        # Simple numeric height
        r"(?P<valeur>\d+(?:[.,]\d+)?)"

        r"|"

        # Unknown height: ?m, ?? m
        r"(?P<valeur_inconnue>\?+)"

    r")"

    r"\s*"
    r"(?P<unite>mm|mètres?|m|ft|')"
    r"\+?"

    # Closing italics: _25m_
    r"_?",

    re.IGNORECASE
)


def extraire_hauteur_debut(texte):
    """
    Extract a height when it appears at the beginning of the text.

    Examples:
        30 m
        30m
        60mm
        60 mètres
        60 mètre
        30/40 m
        30/40m
        30/40 mètres
        30-35 m
        30-35m
        30-35 mètres
        100'
        45m+
        ?m
        ? m
        ? mètres
    """

    texte = texte.strip()

    match = motif_hauteur.match(texte)

    if not match:
        return pd.NA, texte

    # ------------------------------------------------------------
    # Unknown height
    # ------------------------------------------------------------

    if match.group("valeur_inconnue") is not None:
        return "?", texte[match.end():].strip()

    # ------------------------------------------------------------
    # Height expressed as a range
    # ------------------------------------------------------------

    if match.group("valeur_plage") is not None:

        valeur = match.group(
            "valeur_plage"
        ).replace(",", ".")

        return valeur, texte[match.end():].strip()

    # ------------------------------------------------------------
    # Simple numeric height
    # ------------------------------------------------------------

    valeur = float(
        match.group("valeur").replace(",", ".")
    )

    return valeur, texte[match.end():].strip()


# ================================================================
# 5. GRADE DETECTION
# ================================================================

def extraire_cotation_debut(texte):
    """
    Extract a grade when it appears at the beginning of the text.

    Supported examples:

        5
        5a
        5c+
        5.10b
        5.7/5.4R
        5c-6b-5c
        5b/c
        6a/+
        5b/c-6a
        6b/A2/6a
        6a+/A1+/6a
        7a+ ou A0
        1PA puis 7a+
        A1/6a
        A1+/6a
        IV+
        V−
        V+/VI−
        4b puis 5a
        5c/6a puis 4a
        6a(+?)
        6a(+)
        6a+(5c A0)
        6a+/6b+ ?
        Crux 6b
        5b un pas 6a
        6b/c ?
        5c/6a
        5cA0
        5b/3c/II
        5c/4b/3b
    """

    texte = texte.strip()

    if not texte:
        return pd.NA, ""

    # ------------------------------------------------------------
    # American grade
    # ------------------------------------------------------------

    cotation_americaine_classe = (
        r"(?:"
            r"5\.\d+(?:[a-z])?"
            r"|"
            r"[1-5](?:st|nd|rd|th)\s+class"
        r")"
    )

    # ------------------------------------------------------------
    # Numeric / French grade
    # ------------------------------------------------------------

    cotation_libre = (
        r"\d+(?:\.\d+)?"
        r"(?:[a-zA-Z]+)?"
        r"(?:\+{1,3}|-)?"
        r"\??"
    )

    # ------------------------------------------------------------
    # Roman numeral grade
    # ------------------------------------------------------------

    cotation_rome = (
        r"(?:"
            r"(?:IV|V|VI|VII|VIII)"
            r"[+−-]?"
        r")"
    )

    # ------------------------------------------------------------
    # Aid grade
    # ------------------------------------------------------------

    cotation_artif = (
        r"(?:"
        r"\d*P?A\d*(?:[+-])?"
        r"|"
        r"A\d+(?:[+-])?"
        r"|"
        r"C[1-5][+F]?"
        r")"
    )

    # ------------------------------------------------------------
    # Grade "Crux 6b"
    # ------------------------------------------------------------

    cotation_crux = (
        r"Crux\s+"
        + cotation_libre
    )

    # ------------------------------------------------------------
    # Basic element
    # ------------------------------------------------------------

    cotation_element = (
    r"(?:"

        # --------------------------------------------------------
        # Grade with parentheses
        # --------------------------------------------------------

        r"(?:"
            + cotation_libre
            + r"|"
            + cotation_rome
            + r"|"
            + cotation_artif
            + r"|"
            + cotation_americaine_classe
        + r")"

        r"\s*\("

        r"(?:"
            + cotation_libre
            + r"|"
            + cotation_rome
            + r"|"
            + cotation_artif
            + r"|"
            + cotation_americaine_classe
            + r"|"
            r"\+"
            r"|"
            r"\+\?"
        + r")"

        r"(?:"
            r"\s+"
            r"(?:"
                + cotation_libre
                + r"|"
                + cotation_rome
                + r"|"
                + cotation_artif
                + r"|"
                + cotation_americaine_classe
                + r"|"
                r"\+"
                r"|"
                r"\+\?"
            + r")"
        r")*"

        r"\)"

        + r"|"

        # --------------------------------------------------------
        # Grade "Crux 6b"
        # --------------------------------------------------------

        + cotation_crux

        + r"|"

        # --------------------------------------------------------
        # Simple grade
        # --------------------------------------------------------

        + cotation_libre

        + r"|"

        + cotation_rome

        + r"|"

        + cotation_artif

        + r"|"

        + cotation_americaine_classe

        + r"|"

        # --------------------------------------------------------
        # Grade in quotation marks
        # --------------------------------------------------------

        r"\""
        + cotation_libre
        + r"\""

    r")"
    )

    # ------------------------------------------------------------
    # Abbreviated variant after "/"
    #
    # Examples:
    #   5b/c
    #   6a/+
    #   6b/-
    # ------------------------------------------------------------

    cotation_apres_slash = (
        r"(?:"
        + cotation_element
        + r"|"
        r"[a-zA-Z][+-]?"
        r"|"
        r"[+-]"
        + r")"
    )

    # ------------------------------------------------------------
    # Combined grade
    #
    # IMPORTANT:
    # Multiple elements separated by "/" are allowed.
    #
    # Examples:
    #   5c/6a
    #   5b/3c/II
    #   5c/4b/3b
    #   6b/A2/6a
    #   6a+/A1+/6a
    #
    # The following are also retained:
    #   5cA0
    # ------------------------------------------------------------

    cotation_element_compose = (
        cotation_element

        # Attached aid grade:
        #   5cA0
        + r"(?:A\d+(?:[+-])?)?"

        # Multiple elements separated by "/"
        #
        #   5c/6a
        #   5b/3c/II
        #   5c/4b/3b
        #   6b/A2/6a
        + r"(?:"
            r"/"
            + cotation_apres_slash
        + r")*"
    )

    # ------------------------------------------------------------
    # ROMAN NUMERAL GRADE DETECTION
    # ------------------------------------------------------------

    motif_cotation_romaine = re.compile(
        r"^\s*"
        r"(?:"
            r"(?:"
                r"I{1,3}"
                r"|IV"
                r"|V"
                r"|VI"
                r"|VII"
                r"|VIII"
                r"|IX"
                r"|X"
            r")"
            r"[+−-]?"
            r"(?:"
                r"\s*[/,-]\s*"
                r"(?:"
                    r"I{1,3}"
                    r"|IV"
                    r"|V"
                    r"|VI"
                    r"|VII"
                    r"|VIII"
                    r"|IX"
                    r"|X"
                r")"
                r"[+−-]?"
            r")?"
        r")"
        r"\s*$",
        re.IGNORECASE
    )

    # ------------------------------------------------------------
    # Main pattern
    # ------------------------------------------------------------

    motif = re.compile(
        rf"^(?P<difficulty>"
        rf"{cotation_element_compose}"

        # ========================================================
        # Allow a ? after a combined grade
        #
        # Example:
        #     6a+/6b+ ?
        # ========================================================

        rf"\??"

        rf"(?:"
            rf"\s*"
            rf"(?:"

                # ------------------------------------------------
                # Slash
                # ------------------------------------------------

                rf"/{cotation_apres_slash}"

                # ------------------------------------------------
                # Attached grade
                # ------------------------------------------------

                rf"| {cotation_element_compose}"

                # ------------------------------------------------
                # Hyphen
                # ------------------------------------------------

                rf"|-{cotation_element_compose}"

                # ------------------------------------------------
                # Greater-than sign / sequence
                # ------------------------------------------------

                rf"|>{cotation_element_compose}"

                # ------------------------------------------------
                # And
                # ------------------------------------------------

                rf"|&{cotation_element_compose}"

                # ------------------------------------------------
                # Grade + aid grade
                # ------------------------------------------------

                rf"|\s*\+\s*{cotation_artif}\s*\??"

                # ------------------------------------------------
                # Alternatives
                # ------------------------------------------------

                rf"|\s*ou\s*{cotation_element_compose}"
                rf"|\s*or\s*{cotation_element_compose}"

                # ------------------------------------------------
                # And
                # ------------------------------------------------

                rf"|\s*et\s*{cotation_element_compose}"

                # ------------------------------------------------
                # A move of
                # ------------------------------------------------

                rf"|\s*un\s+pas\s+de\s*{cotation_element_compose}"

                # ------------------------------------------------
                # Then
                # ------------------------------------------------

                rf"|puis{cotation_element_compose}"

                # ------------------------------------------------
                # Then, with spaces
                # ------------------------------------------------

                rf"|\s*puis\s*{cotation_element_compose}"

                # ------------------------------------------------
                # A move
                #
                # Example:
                #   5c un pas 6a
                # ------------------------------------------------

                rf"|\s*un\s+pas\s+{cotation_element_compose}"

            rf")"
        rf")*"
        rf")"

        rf"(?=\s|$|[,;:(!])",
        re.IGNORECASE
    )

    # ------------------------------------------------------------
    # Test full Roman-numeral grade
    # ------------------------------------------------------------

    if motif_cotation_romaine.fullmatch(texte):
        return texte, ""

    # ------------------------------------------------------------
    # Test the general pattern
    # ------------------------------------------------------------

    match = motif.match(texte)

    if not match:
        return pd.NA, texte

    difficulty = match.group(
        "difficulty"
    ).strip()

    reste = texte[
        match.end():
    ].strip()

    # ------------------------------------------------------------
    # Clean up separators after the grade
    # ------------------------------------------------------------

    reste = reste.lstrip(
        " ,;:-"
    ).strip()

    return difficulty, reste

# ================================================================
# 6. PARSING A LINE WITHOUT "|"
# ================================================================

def parser_sans_pipe(texte):
    """
    Parse a pitch's first line when it does not contain "|".
    """

    texte = texte.strip()

    # ------------------------------------------------------------
    # Remove the L1 / L# / P1 label
    # ------------------------------------------------------------

    match = motif_longueur.match(texte)

    if not match:
        return pd.NA, pd.NA, pd.NA, texte

    label = match.group("label")

    reste = texte[match.end():].strip()

    # Remove separators immediately after the label
    reste = reste.lstrip(" :;,|-").strip()

    # ------------------------------------------------------------
    # Special case: L#~
    # ------------------------------------------------------------

    if texte[match.end():].lstrip().startswith("~"):

        reste = texte[match.end():].lstrip()
        reste = reste[1:].strip()

        return label, pd.NA, pd.NA, reste

    # ------------------------------------------------------------
    # Nothing after the label
    # ------------------------------------------------------------

    if not reste:
        return label, pd.NA, pd.NA, ""

    # ------------------------------------------------------------
    # Case 1:
    # height first
    # ------------------------------------------------------------

    height, apres_hauteur = extraire_hauteur_debut(reste)

    if pd.notna(height):

        apres_hauteur = apres_hauteur.lstrip(
            " ,;:-"
        ).strip()

        difficulty, description = (
            extraire_cotation_debut(
                apres_hauteur
            )
        )

        if pd.isna(difficulty):
            description = apres_hauteur

        return (
            label,
            height,
            difficulty,
            description
        )

    # ------------------------------------------------------------
    # Case 2:
    # grade first
    # ------------------------------------------------------------

    difficulty, apres_cotation = (
        extraire_cotation_debut(reste)
    )

    if pd.notna(difficulty):

        height, description = (
            extraire_hauteur_debut(
                apres_cotation.lstrip(
                    " ,;:-"
                )
            )
        )

        if pd.isna(height):
            description = apres_cotation

        return (
            label,
            height,
            difficulty,
            description
        )

    # ------------------------------------------------------------
    # Case 3:
    # unrecognized structure
    # ------------------------------------------------------------

    return (
        label,
        pd.NA,
        pd.NA,
        reste
    )


# ================================================================
# 7. GEAR DETECTION
# ================================================================

motif_equipement = re.compile(
    r"^\s*"
    r"(?:"

        # --------------------------------------------------------
        # No gear
        # --------------------------------------------------------
        r"Pas\s+d['’]équipement"

        r"|"

        r"Non\s+équipé"

        r"|"

        # --------------------------------------------------------
        # One or more quantified gear items
        #
        # Examples:
        #   8 pt
        #   8 pts
        #   8-10 pt
        #   8-10 pts
        #   8-10pt
        #   8/10 pts
        #   4 goujons + 1 burillo
        #   8 goujons + 1 piton rouge + 1 lunule
        # --------------------------------------------------------
        r"\d+(?:\s*-\s*\d+|\s*/\s*\d+)?\s*"
        r"(?:"
            r"pts?"
            r"|points?"
            r"|pitons?"
            r"|spits?"
            r"|broches?"
            r"|goujons?"
            r"|g"
            r"|p"
            r"|friends?"
            r"|coinceurs?"
            r"|plaquettes?"
            r"|lunules?"
            r"|burillo"
        r")"

        # --------------------------------------------------------
        # Optional note after the first gear item
        # --------------------------------------------------------
        r"(?:"
            r"\s+au\s+départ"
            r"|"
            r"\s+(?:rouge|blanc|noir|bleu|jaune|vert)"
        r")?"

        # --------------------------------------------------------
        # Additional gear
        # --------------------------------------------------------
        r"(?:"
            r"\s*"
            r"(?:"
                r",\s*"
                r"|"
                r"et\s+"
                r"|"
                r"\+\s*"
                r"|"
                r"\s+"
            r")"
            r"(?:"
                r"\d+(?:\s*-\s*\d+|\s*/\s*\d+)?"
                r"|un"
                r"|une"
                r"|deux"
            r")"
            r"\s*"
            r"(?:"
                r"pts?"
                r"|points?"
                r"|pitons?"
                r"|spits?"
                r"|broches?"
                r"|goujons?"
                r"|g"
                r"|friends?"
                r"|coinceurs?"
                r"|plaquettes?"
                r"|lunules?"
                r"|burillo"
            r")"
            r"(?:"
                r"\s+au\s+départ"
                r"|"
                r"\s+(?:rouge|blanc|noir|bleu|jaune|vert)"
            r")?"
        r")*"

    r")"
    r"\s*$",
    re.IGNORECASE
)


equipements_specifiques = {
    "1u",
    "1r",
    "1u 4p 1f",
    "1p 1u",
    "2u"
}



# ================================================================
# 7bis. NOTE ATTACHED TO THE GRADE: "6a (1 pas)", "6b (5c/A0)"
# ================================================================
#
# A short parenthetical immediately after the grade, in the same field,
# comments on the grade. It is stored in `difficulty_note`
# (without the parentheses) instead of cluttering the description.

motif_note_cotation = re.compile(
    r"^\((?P<note>[^()]{1,40})\)\s*(?P<reste>.*)$",
    re.DOTALL
)


def separer_note_cotation(reste):
    """
    Return (note, remainder_without_note). note is pd.NA if none is present.
    """
    if not isinstance(reste, str):
        return pd.NA, reste

    m = motif_note_cotation.match(reste.strip())

    if not m:
        return pd.NA, reste

    return m.group("note").strip(), m.group("reste").strip()


# ================================================================
# 7ter. ABBREVIATED GEAR: "1br", "1g 1p", "9 pnts", "Spits et pitons"
# ================================================================
#
# A field is considered gear if all its words belong to the
# gear vocabulary AND it contains at least one indicator:
#   - a gear name (spits, pitons, bolts, ...)
#   - or an abbreviation preceded by a number (1br, 3g, 9pnts, ?g)
#   - or "nothing in place"
#
# The field is kept in the `equipment` column (not discarded).

UNITES_EQUIP_COURTES = {
    "g", "p", "c", "u", "l", "r", "f", "s", "br",
    "pt", "pts", "pnt", "pnts", "ts", "ƿ",
}

MOTS_EQUIP = {
    "goujon", "goujons", "gouj",
    "point", "points",
    "spit", "spits",
    "piton", "pitons",
    "lunule", "lunules",
    "broche", "broches",
    "friend", "friends",
    "coinceur", "coinceurs",
    "plaquette", "plaquettes",
    "anneau", "anneaux",
    "burin", "burins", "burillo",
    "dégaine", "dégaines",
    "clou", "clous",
    "équipement", "équipée", "équipé", "équipées", "équipés",
}

MOTS_LIAISON_EQUIP = {
    "et", "ou", "avec", "au", "aux", "départ", "de", "d", "du", "des",
    "en", "place", "sur", "un", "une", "deux", "quelques", "plusieurs",
    "nombreux", "nombreuses", "beaucoup", "aucun", "aucune", "rien",
    "peu", "non", "bien", "très", "à", "équiper", "relais",
    "rouge", "rouges", "blanc", "blancs", "noir", "bleu", "jaune", "vert",
}

motif_jeton_equip = re.compile(
    r"\d+(?:[.,]\d+)?|\?+|[a-zà-ÿœƿ]+|#",
    re.IGNORECASE
)

motif_nombre_equip = re.compile(r"\d+(?:[.,]\d+)?|\?+")


def est_equipement_abrege(champ):

    s = champ.strip().lower().replace("’", "'")

    if not s or len(s) > 60:
        return False

    # No characters other than words, numbers, and simple punctuation
    residu = motif_jeton_equip.sub("", s)

    if re.search(r"[^\s,.;:+&/()'*\-]", residu):
        return False

    jetons = motif_jeton_equip.findall(s)

    if not jetons:
        return False

    indicateur = False
    unites = []

    for i, j in enumerate(jetons):

        if motif_nombre_equip.fullmatch(j) or j == "#":
            continue

        if j in MOTS_EQUIP:
            indicateur = True
            continue

        # "pas de point(s)": "pas" is accepted only before "de"/"d"
        if j == "pas":
            if i + 1 < len(jetons) and jetons[i + 1] in ("de", "d"):
                continue
            return False

        if j in MOTS_LIAISON_EQUIP:
            continue

        if j in UNITES_EQUIP_COURTES:

            precedent = jetons[i - 1] if i > 0 else ""

            if motif_nombre_equip.fullmatch(precedent):
                unites.append(j)
                continue

            return False

        return False

    # "5c" and "6c" alone are grades, but "5 coinceurs" is not
    if unites and not indicateur:

        if len(unites) == 1 and unites[0] == "c":
            return False

        return True

    if indicateur:
        return True

    # "Rien", "Rien en place"
    return "rien" in jetons and set(jetons) <= {"rien", "en", "place"}


# ================================================================
# 7quater. ADDITIONAL GRADE: "6a", "C2", "puis III", "/ A0"
# ================================================================
#
# When `difficulty` is already populated and another field consists entirely
# of a grade, it is stored in `difficulty_extra`.
# Bare numbers ("25", "8") are never treated as grades here.

motif_connecteur_cotation = re.compile(
    r"^(?:puis|ou|et|/|>|&)\s*",
    re.IGNORECASE
)


def extraire_cotation_supplementaire(champ):

    champ = champ.strip()

    texte = motif_connecteur_cotation.sub("", champ, count=1)

    cotation, reste = extraire_cotation_debut(texte)

    if pd.isna(cotation) or reste:
        return pd.NA

    est_nombre = re.fullmatch(r"\d+(?:[.,]\d+)?", cotation)

    est_romain = re.fullmatch(
        r"(?i)[IVX]+[+−-]?(?:\s*[/,-]\s*[IVX]+[+−-]?)*",
        cotation
    )

    a_chiffre = re.search(r"\d", cotation)

    if est_romain or (a_chiffre and not est_nombre):
        return champ

    return pd.NA


# ================================================================
# 8. PARSING A LINE WITH "|"
# ================================================================

def parser_avec_pipe(texte):
    """
    Parse pitches that use "|".

    Examples:

        L# | 6b+ | 30m | description
        L# | 30m | 6b+ | description

        L# | 6b/A2/6a | 45 m | description
        L# | 6a+/A1+/6a | 50 m | description

        L# | 6b+/6a+/A1/5c | | description
        L# | 6a | | description
        L# | A1/6a | | description

        L# | 6a | 40 m | 11 pts | description

    Fields corresponding to:
        - a height
        - a grade
        - gear
    are treated as metadata.
    """

    match = motif_longueur.match(texte)

    if not match:
        return pd.NA, pd.NA, pd.NA, texte, pd.NA, pd.NA, pd.NA

    label = match.group("label")

    reste = texte[match.end():].strip()

    # ------------------------------------------------------------
    # Remove the first "|"
    # ------------------------------------------------------------

    if reste.startswith("|"):
        reste = reste[1:]

    # ------------------------------------------------------------
    # Split on "|"
    # ------------------------------------------------------------

    champs = [
        champ.strip()
        for champ in reste.split("|")
    ]

    # ------------------------------------------------------------
    # Remove empty fields at the end of the line
    # ------------------------------------------------------------

    while champs and champs[-1] == "":
        champs.pop()

    height = pd.NA
    difficulty = pd.NA
    note_difficulte = pd.NA
    equipements = []
    cotations_extra = []

    champs_description = []

    # ------------------------------------------------------------
    # Analyze all fields
    # ------------------------------------------------------------

    for champ in champs:

        if not champ:
            continue

        # --------------------------------------------------------
        # 1. Height
        # --------------------------------------------------------

        hauteur, reste_hauteur = (
            extraire_hauteur_debut(champ)
        )

        if pd.notna(hauteur) and pd.isna(height):

            height = hauteur

            if reste_hauteur:
                champs_description.append(
                    reste_hauteur
                )

            continue

        # --------------------------------------------------------
        # 2. Difficulty
        # --------------------------------------------------------

        cotation, reste_cotation = (
            extraire_cotation_debut(champ)
        )

        if pd.notna(cotation) and pd.isna(difficulty):

            difficulty = cotation

            # Parenthetical attached to the grade: "6a (1 pas)"
            # -> grade note, not a description
            note, reste_cotation = separer_note_cotation(
                reste_cotation
            )

            if pd.notna(note):
                note_difficulte = note

            if reste_cotation:
                champs_description.append(
                    reste_cotation
                )

            continue

        # --------------------------------------------------------
        # 3. Gear
        # --------------------------------------------------------

        if motif_equipement.fullmatch(champ):
            equipements.append(champ)
            continue

        if champ.strip().lower() in equipements_specifiques:
            equipements.append(champ)
            continue

        if est_equipement_abrege(champ):
            equipements.append(champ)
            continue

        # --------------------------------------------------------
        # 3bis. Additional grade (`difficulty` already populated)
        # --------------------------------------------------------

        if pd.notna(difficulty):

            extra = extraire_cotation_supplementaire(champ)

            if pd.notna(extra):
                cotations_extra.append(extra)
                continue

        # --------------------------------------------------------
        # 4. Everything else = description
        # --------------------------------------------------------

        champs_description.append(champ)

    # ------------------------------------------------------------
    # Final description
    # ------------------------------------------------------------

    description = " | ".join(
        champ.strip()
        for champ in champs_description
        if champ.strip()
    ).strip()

    return (
        label,
        height,
        difficulty,
        description,
        note_difficulte,
        " | ".join(equipements) if equipements else pd.NA,
        " | ".join(cotations_extra) if cotations_extra else pd.NA
    )


# ================================================================
# 9. GENERAL FIRST-LINE PARSER
# ================================================================

def parser_en_tete_longueur(texte):
    """
    Automatically choose between:
        - parser with "|"
        - parser without "|"

    Return: label, height, difficulty, description,
               note_difficulte, equipement, cotations_supplementaires
    """

    if "|" in texte:

        return parser_avec_pipe(texte)

    else:

        # parser_sans_pipe does not produce a grade note
        return (*parser_sans_pipe(texte), pd.NA, pd.NA, pd.NA)


# ================================================================
# 10. JUNCTION LINES TO IGNORE
# ================================================================

def est_ligne_jonction(texte):
    """
    Detect standalone lines describing a junction
    between two pitches.

    These lines must not be attached to the preceding
    pitch.

    Examples:

        Short junction.
        Long junction.
        Long junction on the right throughout, despite the cables on the left.
        Short junction on the left.
        30 m descending junction (2 bolts).

    IMPORTANT:
    do not remove the word "junction" from inside
    a real pitch description.
    """

    texte = texte.strip()

    if not texte:
        return False

    # ------------------------------------------------------------
    # Lines beginning with "Jonction" are considered
    # standalone junction lines.
    # ------------------------------------------------------------

    return bool(
        re.match(
            r"(?i)^jonction\b",
            texte
        )
    )


# ================================================================
# 11. CLEANING THE BLOCK AFTER THE HEADER
# ================================================================

def nettoyer_suite_bloc(suite):
    """
    Clean the lines following a pitch header.

    Remove:
        - Markdown headings ## / ###
        - standalone junction lines

    Keep:
        - actual descriptions
        - regular text lines
    """

    if not suite:
        return ""

    lignes = suite.splitlines()

    lignes_conservees = []

    for ligne in lignes:

        ligne_strip = ligne.strip()

        if not ligne_strip:
            continue

        # --------------------------------------------------------
        # Section headings are never a description.
        # --------------------------------------------------------

        if re.match(
            r"^#{2,3}\s+",
            ligne_strip
        ):
            continue

        # --------------------------------------------------------
        # Standalone junction lines are ignored.
        # --------------------------------------------------------

        if est_ligne_jonction(ligne_strip):
            continue

        lignes_conservees.append(
            ligne_strip
        )

    return "\n".join(
        lignes_conservees
    ).strip()


# ================================================================
# 11bis. RECOVERING HEIGHTS WITHOUT A UNIT ("35" instead of "35 m")
# ================================================================
#
# Principle: for a route, if previous pitches had a
# height (with a unit) at position x in the header "L# | ... | ... |",
# and the current pitch has a bare number (10, 20, 35, 30/40...)
# at that same position x, that number is a height whose
# author omitted the unit. Append " m" to the field BEFORE parsing,
# so the existing parser naturally stores it in
# `height` (rather than `difficulty`, where a bare "35" would otherwise end up).

# Safeguard: a bare number below this threshold could be a
# grade (e.g. "3", "5", "6"), not a height.
HAUTEUR_NUE_MIN = 8
HAUTEUR_NUE_MAX = 300

motif_nombre_nu = re.compile(
    r"(?P<a>\d+(?:[.,]\d+)?)"
    r"(?:\s*[/-]\s*(?P<b>\d+(?:[.,]\d+)?))?"
)


def decouper_champs_pipe(en_tete):
    """
    Return (prefixe_label, champs) for a header containing "|",
    or None if the header cannot be used.
    The splitting is identical to that in parser_avec_pipe,
    so field indices are consistent.
    """
    if "|" not in en_tete:
        return None

    match = motif_longueur.match(en_tete)

    if not match:
        return None

    prefixe = en_tete[:match.end()].rstrip()
    reste = en_tete[match.end():].strip()

    if reste.startswith("|"):
        reste = reste[1:]

    champs = [c.strip() for c in reste.split("|")]

    return prefixe, champs


def position_hauteur_pipe(en_tete):
    """
    Index of the field containing a height WITH a unit, or None.
    """
    decoupe = decouper_champs_pipe(en_tete)

    if decoupe is None:
        return None

    _, champs = decoupe

    for i, champ in enumerate(champs):

        if not champ:
            continue

        hauteur, _ = extraire_hauteur_debut(champ)

        if pd.notna(hauteur):
            return i

    return None


def corriger_hauteur_manquante(en_tete, position_ref):
    """
    If the header has no recognized height, but a bare number
    appears at the reference position, append " m" to that field.

    Return (en_tete_corrige, True/False).
    """
    if position_ref is None:
        return en_tete, False

    decoupe = decouper_champs_pipe(en_tete)

    if decoupe is None:
        return en_tete, False

    prefixe, champs = decoupe

    # A height is already present somewhere: nothing to do
    if position_hauteur_pipe(en_tete) is not None:
        return en_tete, False

    if position_ref >= len(champs):
        return en_tete, False

    champ = champs[position_ref]

    m = motif_nombre_nu.fullmatch(champ)

    if not m:
        return en_tete, False

    valeurs = [
        float(v.replace(",", "."))
        for v in (m.group("a"), m.group("b"))
        if v is not None
    ]

    if not all(
        HAUTEUR_NUE_MIN <= v <= HAUTEUR_NUE_MAX for v in valeurs
    ):
        return en_tete, False

    champs[position_ref] = champ + " m"

    return prefixe + " | " + " | ".join(champs), True


# ================================================================
# 11ter. HEIGHT FORMAT TRACEABILITY
# ================================================================

def analyser_format_hauteur(en_tete):
    """
    For a header with a recognized height, return:
        (approx, variante)

        approx: True if the height is approximate (~, <, >, approx.)
        variante: True if it is written in an unusual form
                   (_25m_, ~30 m, Env. 30 m, ?? m ...)
    """
    decoupe = decouper_champs_pipe(en_tete)

    if decoupe is not None:

        candidats = decoupe[1]

    else:

        match = motif_longueur.match(en_tete)

        if not match:
            return False, False

        reste = en_tete[match.end():].strip().lstrip(" :;,|-").strip()

        candidats = [reste]

        _, apres_cotation = extraire_cotation_debut(reste)

        candidats.append(
            str(apres_cotation).lstrip(" ,;:-")
        )

    for champ in candidats:

        m = motif_hauteur.match(champ.strip())

        if m:

            approx = m.group("approx") is not None

            variante = (
                approx
                or champ.strip().startswith("_")
                or (m.group("valeur_inconnue") or "?") != "?"
            )

            return approx, variante

    return False, False


# ================================================================
# 11quater. TABLE HEADERS MISTAKEN FOR PITCHES
# ================================================================
#
# Examples:
#     L# | = | Cot. | Long. | Points | Commentaire | Relais
#     L# | = | Relais
#
# These are column headings; they do not describe a pitch.

MOTS_ENTETE_TABLEAU = {
    "=", "#", "n°",
    "cot.", "cot", "cotation", "diff.", "difficulté",
    "long.", "long", "longueur", "length", "hauteur",
    "points", "pts", "équipement", "equipement",
    "commentaire", "commentaires", "description", "descriptif",
    "relais",
}


def est_entete_tableau(en_tete):
    decoupe = decouper_champs_pipe(en_tete)

    if decoupe is None:
        return False

    champs = [c.strip().lower() for c in decoupe[1] if c.strip()]

    if not champs:
        return False

    # First field "=": table-header marker
    if champs[0] == "=":
        return True

    return all(c in MOTS_ENTETE_TABLEAU for c in champs)


# ================================================================
# 11quinquies. FALLBACK: ROUTE WHERE NO HEIGHT HAS A UNIT
# ================================================================
#
# Case: the author never wrote "m" ("L# | III | 20 | ...").
# There is then no reference pitch. Infer the position
# of the height if, across ALL pitches on the route, only one
# position contains a plausible bare number (at least 2 pitches,
# median >= 15 to avoid confusing it with a points count).

HAUTEUR_SECOURS_MEDIANE_MIN = 15


def valeurs_hauteur_nue(champ):

    m = motif_nombre_nu.fullmatch(champ.strip())

    if not m:
        return None

    valeurs = [
        float(v.replace(",", "."))
        for v in (m.group("a"), m.group("b"))
        if v is not None
    ]

    if all(HAUTEUR_NUE_MIN <= v <= HAUTEUR_NUE_MAX for v in valeurs):
        return valeurs

    return None


def position_hauteur_secours(en_tetes):

    decoupes = []

    for en_tete in en_tetes:

        if est_entete_tableau(en_tete):
            continue

        if position_hauteur_pipe(en_tete) is not None:
            return None

        decoupe = decouper_champs_pipe(en_tete)

        if decoupe is None:
            return None

        decoupes.append(decoupe[1])

    if len(decoupes) < 2:
        return None

    candidats = None

    for champs in decoupes:

        positions = {
            i for i, champ in enumerate(champs)
            if valeurs_hauteur_nue(champ) is not None
        }

        candidats = (
            positions if candidats is None
            else candidats & positions
        )

    if not candidats or len(candidats) != 1:
        return None

    position = next(iter(candidats))

    valeurs = sorted(
        max(valeurs_hauteur_nue(champs[position]))
        for champs in decoupes
    )

    mediane = valeurs[len(valeurs) // 2]

    if mediane < HAUTEUR_SECOURS_MEDIANE_MIN:
        return None

    return position


# ================================================================
# 12. COMPLETE SPLIT OF MULTI-PITCH ROUTES
# ================================================================

def decouper_descriptions_multi(dataset):

    lignes = []

    nb_entetes_tableau = 0

    for _, ligne in dataset.iterrows():

        # --------------------------------------------------------
        # Added columns
        # --------------------------------------------------------

        ligne_base = ligne.copy()

        ligne_base["longueur"] = pd.NA
        ligne_base["height"] = pd.NA
        ligne_base["difficulty"] = pd.NA
        ligne_base["height_corrigee"] = False
        ligne_base["height_variante"] = False
        ligne_base["height_approx"] = False
        ligne_base["difficulty_note"] = pd.NA
        ligne_base["difficulty_extra"] = pd.NA
        ligne_base["equipment"] = pd.NA

        # --------------------------------------------------------
        # Non-multi-pitch routes
        # --------------------------------------------------------

        if ligne["climbing_outdoor_type"] != "multi":

            lignes.append(ligne_base)
            continue

        description = ligne["description"]

        if not isinstance(description, str):
            lignes.append(ligne_base)
            continue

        if not description.strip():
            lignes.append(ligne_base)
            continue

        # ========================================================
        # 1. FIND ALL REAL PITCHES
        # ========================================================

        correspondances = list(
            motif_longueur.finditer(description)
        )

        if not correspondances:

            lignes.append(ligne_base)
            continue

        # ========================================================
        # 2. FIND SECTION HEADINGS
        # ========================================================

        sections = list(
            motif_titre_section.finditer(
                description
            )
        )

        # ========================================================
        # 3. FIND L#~ MARKERS
        # ========================================================

        motif_fin_longueur_tilde = re.compile(
            r"(?mi)^[ \t]*L#[ \t]*~"
        )

        fins_tilde = list(
            motif_fin_longueur_tilde.finditer(
                description
            )
        )

        # ========================================================
        # 4. PROCESS EACH PITCH
        # ========================================================

        # Height position observed in previous pitches.
        # Initially: possible fallback if no pitch on the route has
        # a height with a unit (see position_hauteur_secours).
        en_tetes_voie = [
            description[m.start():].split("\n", 1)[0].strip()
            for m in correspondances
        ]

        position_hauteur_ref = position_hauteur_secours(en_tetes_voie)

        for numero, correspondance in enumerate(
            correspondances
        ):

            debut = correspondance.start()

            bornes = []

            # ----------------------------------------------------
            # Boundary 1:
            # next real pitch
            # ----------------------------------------------------

            if numero + 1 < len(correspondances):

                bornes.append(
                    correspondances[numero + 1].start()
                )

            # ----------------------------------------------------
            # Boundary 2:
            # next L#~ marker
            # ----------------------------------------------------

            for m in fins_tilde:

                if m.start() > debut:

                    bornes.append(
                        m.start()
                    )

                    break

            # ----------------------------------------------------
            # Boundary 3:
            # next Markdown heading
            #
            # IMPORTANT:
            #
            # The heading itself will never be part of the block
            # for the pitch.
            # ----------------------------------------------------

            for section in sections:

                if section.start() > debut:

                    bornes.append(
                        section.start()
                    )

                    break

            # ----------------------------------------------------
            # Determine the nearest boundary
            # ----------------------------------------------------

            if bornes:

                fin = min(bornes)

            else:

                fin = len(description)

            # ----------------------------------------------------
            # Extract the block
            # ----------------------------------------------------

            bloc = description[
                debut:fin
            ].strip()

            if not bloc:
                continue

            # ----------------------------------------------------
            # First line = pitch header
            # ----------------------------------------------------

            lignes_bloc = bloc.splitlines()

            if not lignes_bloc:
                continue

            en_tete = lignes_bloc[0].strip()

            # ----------------------------------------------------
            # Everything after the first line
            # ----------------------------------------------------

            suite = "\n".join(
                lignes_bloc[1:]
            ).strip()

            # ----------------------------------------------------
            # Clean up the remainder
            # ----------------------------------------------------

            suite = nettoyer_suite_bloc(
                suite
            )

            # ----------------------------------------------------
            # Parse the header
            # ----------------------------------------------------

            # ----------------------------------------------------
            # Table header (column headings): ignore it
            # ----------------------------------------------------

            if est_entete_tableau(en_tete):
                nb_entetes_tableau += 1
                continue

            # ----------------------------------------------------
            # Recover a height whose unit was omitted
            # ----------------------------------------------------

            en_tete, height_corrigee = corriger_hauteur_manquante(
                en_tete,
                position_hauteur_ref
            )

            # Remember the height position for
            # subsequent pitches on the same route
            pos = position_hauteur_pipe(en_tete)

            if pos is not None:
                position_hauteur_ref = pos

            (
                longueur,
                height,
                difficulty,
                description_premiere_ligne,
                difficulty_note,
                equipement,
                difficulty_extra
            ) = parser_en_tete_longueur(
                en_tete
            )

            # Track the height format
            height_approx, height_variante = False, False

            if pd.notna(height):
                height_approx, height_variante = (
                    analyser_format_hauteur(en_tete)
                )

            # ----------------------------------------------------
            # Create the new row
            # ----------------------------------------------------

            nouvelle_ligne = ligne.copy()

            nouvelle_ligne["longueur"] = longueur
            nouvelle_ligne["height"] = height
            nouvelle_ligne["difficulty"] = difficulty
            nouvelle_ligne["height_corrigee"] = height_corrigee
            nouvelle_ligne["height_variante"] = height_variante
            nouvelle_ligne["height_approx"] = height_approx
            nouvelle_ligne["difficulty_note"] = difficulty_note
            nouvelle_ligne["difficulty_extra"] = difficulty_extra
            nouvelle_ligne["equipment"] = equipement

            # ----------------------------------------------------
            # Final description
            # ----------------------------------------------------

            morceaux_description = []

            if description_premiere_ligne:

                morceaux_description.append(
                    description_premiere_ligne
                )

            if suite:

                morceaux_description.append(
                    suite
                )

            nouvelle_ligne["description"] = "\n".join(
                morceaux_description
            ).strip()

            lignes.append(
                nouvelle_ligne
            )

    print(
        f"{nb_entetes_tableau} table headers ignored"
    )

    return pd.DataFrame(
        lignes
    ).reset_index(drop=True)


# ================================================================
# 13. RUN
# ================================================================

df_par_longueur = decouper_descriptions_multi(df)

33 en-têtes de tableau ignorés


In [ ]:
# Save the results so the splitting does not need to be rerun each time
df_par_longueur.to_csv("voies_par_longueur.csv", index=False, encoding="utf-8")

# df_par_longueur = pd.read_csv("voies_par_longueur.csv", encoding="utf-8")
df_par_longueur

,document_id,title,title_prefix,quality,activities,climbing_outdoor_type,global_rating,engagement_rating,risk_rating,equipment_rating,...,description,longueur,height,difficulty,height_corrigee,height_variante,height_approx,difficulty_note,difficulty_extra,equipment
0,1948187,Triangle des Coutchets - Brouillard d'été,Pic des Coutchets,draft,rock_climbing,multi,NaN,NaN,NaN,NaN,...,A gauche de Aquim,NaN,NaN,NaN,False,False,False,NaN,NaN,NaN
1,1948176,Triangle des Coutchets - Soleil d'automne,Pic des Coutchets,medium,rock_climbing,multi,TD-,NaN,NaN,P2,...,"Remonter la petite fissure, aller un peu à gau...",L1,55.0,IV+,False,False,False,NaN,NaN,NaN
2,1948176,Triangle des Coutchets - Soleil d'automne,Pic des Coutchets,medium,rock_climbing,multi,TD-,NaN,NaN,P2,...,Monter vers une dalle puis un dièdre à gauche ...,L2,45.0,5c,False,False,False,NaN,NaN,NaN
3,1948176,Triangle des Coutchets - Soleil d'automne,Pic des Coutchets,medium,rock_climbing,multi,TD-,NaN,NaN,P2,...,Rejoindre au dessus le dièdre fissure et le su...,L3,35.0,5c,False,False,False,NaN,NaN,NaN
4,1948176,Triangle des Coutchets - Soleil d'automne,Pic des Coutchets,medium,rock_climbing,multi,TD-,NaN,NaN,P2,...,"Continuer dans le dièdre, partir à gauche dans...",L4,35.0,5c+,False,False,False,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
71983,53893,Voie Directe,Miroir d'Argentine,fine,rock_climbing,multi,D,II,NaN,P2,...,"Monter dans un système de fissures, assez vert...",L#,NaN,4a,False,False,False,NaN,NaN,NaN
71984,53893,Voie Directe,Miroir d'Argentine,fine,rock_climbing,multi,D,II,NaN,P2,...,Suivre la fine fissure qui part diagonalement ...,L#,NaN,5b,False,False,False,NaN,NaN,NaN
71985,53893,Voie Directe,Miroir d'Argentine,fine,rock_climbing,multi,D,II,NaN,P2,...,"Franchir un petit surplomb (1 spit, 4b), puis ...",L#,NaN,5b,False,False,False,NaN,NaN,NaN
71986,53893,Voie Directe,Miroir d'Argentine,fine,rock_climbing,multi,D,II,NaN,P2,...,Monter plus ou moins tout droit dans la dalle ...,L#,NaN,3c-4a,False,False,False,NaN,NaN,NaN


## 5. Second data cleaning

In [3]:
# Remove all pitches without a description (NaN, empty string, or whitespace)
df_par_longueur = df_par_longueur[
    df_par_longueur['description'].fillna('').str.strip() != ''
].reset_index(drop=True)

## 6. Grade standardization

In [ ]:
"""
Classify the `difficulty` column by grading system.

Add two columns to the DataFrame:

    systeme_cotation : francaise | americaine | romaine | nombre_nu |
                       artificielle | composee | annotee | autre | manquante
    composition      : for combined grades only, the systems
                       that make them up, in order (e.g. "francaise+artificielle")

Category definitions
-------------------------
francaise     3a, 5c+, 6B             (a single grade, with no surrounding text)
americaine    5.9, 5.10a, 5.11-       (YDS)
romaine       IV, V+, VI-             (UIAA)
nombre_nu     3, 4+, 5-               (neither Roman nor letter-based: ambiguous)
artificielle  A0, A2+, C1, C2F        (aid / clean aid, alone)
composee      at least two grades: 5c/6a, 6a/b, 6a/+, 6b>A0, V+/6a,
              6a puis 5c, 6aA0, 2/3, II/III ...
annotee       a single grade + surrounding text: 6a?, 6b(+), 6a++, 6a 30,
              HVS 5a ...  (review case by case)
autre         no recognizable grade: 0p, 2g, 4th, 10a ...
manquante     empty value
"""


# ---------------------------------------------------------------------------
# Elementary grade detection
# ---------------------------------------------------------------------------

# The order of alternatives matters: test the most specific first.
MOTIF_TOKEN = re.compile(
    r"""
      (?P<americaine>   (?<![\w.])5\.\d{1,2}[abcd]?[+-]?(?![\w]) )
    | (?P<artificielle> (?<![A-Z])(?:A[0-5O]\+?|C[1-5][+F]?)(?![0-9]) )
    | (?P<francaise>    (?<![\w.])[1-9](?:[abc]|[ABC](?![0-9]))\+?(?![a-z]) )
    | (?P<romaine>      (?<![A-Za-z])(?:VIII|VII|VI|IV|V|IX|X|III|II|I)[+-]?(?![A-Za-z]) )
    """,
    re.VERBOSE,
)

# Abbreviated French grade ranges: 6a/b, 6a+/b, 6a/+, 6ab
MOTIF_FOURCHETTE_ABREGEE = re.compile(
    r"[1-9][abc]\+?(?:/(?:\+|[abc]\+?)|[abc]\+?)"
)

# Bare numbers: 3, 4+, 5-
MOTIF_NOMBRE_NU = re.compile(r"\d[+-]?")

# Ranges of bare numbers: 2/3, 3-4, 4+/5-, 5/5+, 4 puis 3
MOTIF_NOMBRES_NUS_COMPOSES = re.compile(
    r"\d[+-]?(?:\s*(?:[/,>-]|ou|puis|et)\s*\d[+-]?)+"
)


SEPARATEUR = r"(?:[/,>-]|ou|puis|et)"
MOTIF_NOMBRE_AVANT = re.compile(r"\d[+-]?\s*" + SEPARATEUR + r"\s*")
MOTIF_NOMBRE_APRES = re.compile(r"\s*" + SEPARATEUR + r"\s*\d[+-]?")


def _normaliser(valeur):
    return (
        str(valeur)
        .replace("−", "-")   
        .replace("–", "-")   
        .strip()
    )


def classifier_cotation(valeur):
    """
    Return (systeme_cotation, composition) for a `difficulty` value.
    """
    if pd.isna(valeur) or str(valeur).strip() == "":
        return "manquante", pd.NA

    s = _normaliser(valeur)

    # 1. Abbreviated French grade ranges (6a/b, 6a/+, 6ab) -> composées
    if MOTIF_FOURCHETTE_ABREGEE.fullmatch(s):
        return "composee", "francaise+francaise"

    # 2. Elementary grades present in the string
    tokens = [
        (m.lastgroup, m.span()) for m in MOTIF_TOKEN.finditer(s)
    ]

    # 2a. Multiple grades -> combined
    if len(tokens) >= 2:
        return "composee", "+".join(systeme for systeme, _ in tokens)

    # 2b. A single grade
    if len(tokens) == 1:
        systeme, (debut, fin) = tokens[0]

        if (debut, fin) == (0, len(s)):
            return systeme, pd.NA       # the string IS the grade

        avant, apres = s[:debut], s[fin:]

        # Bare number joined by a separator: 5+/6a, 4/5c, 6a puis 4
        if not apres and MOTIF_NOMBRE_AVANT.fullmatch(avant):
            return "composee", "nombre_nu+" + systeme

        if not avant and MOTIF_NOMBRE_APRES.fullmatch(apres):
            return "composee", systeme + "+nombre_nu"

        return "annotee", pd.NA         # grade + surrounding text

    # 3. No letter / Roman numeral / YDS / aid grade
    if MOTIF_NOMBRE_NU.fullmatch(s):
        return "nombre_nu", pd.NA

    if MOTIF_NOMBRES_NUS_COMPOSES.fullmatch(s):
        return "composee", "nombre_nu+nombre_nu"

    return "autre", pd.NA


def classifier_cotations(df, colonne="difficulty"):
    """
    Add `systeme_cotation` and `composition` to the DataFrame (copy).
    Classify each distinct value only once (fast).
    """
    df = df.copy()

    valeurs = df[colonne].drop_duplicates()

    table = pd.DataFrame(
        [classifier_cotation(v) for v in valeurs],
        index=valeurs,
        columns=["systeme_cotation", "composition"],
    )

    # NaN values cannot be indexed like the others, so handle them separately
    cles = df[colonne]
    df["systeme_cotation"] = cles.map(table["systeme_cotation"])
    df["composition"] = cles.map(table["composition"])
    df.loc[cles.isna(), "systeme_cotation"] = "manquante"

    return df


# ---------------------------------------------------------------------------
# Usage
# ---------------------------------------------------------------------------
if __name__ == "__main__":

    df = df_par_longueur
    df = classifier_cotations(df)


In [5]:
df['systeme_cotation'].value_counts()

systeme_cotation
francaise       42861
manquante        5589
composee         5225
romaine          3549
nombre_nu        3313
americaine       1112
artificielle      409
annotee           334
autre             123
Name: count, dtype: int64

In [ ]:
# Remove all rows without a grade (`difficulty`), as they cannot be analyzed correctly.

df = df[df['systeme_cotation'] != 'manquante'].reset_index(drop=True)
# df[['difficulty', 'systeme_cotation', 'composition']]

# Keep only pitches with a non-combined French grade, as this is currently the most common and consistent system.
df = df[df['systeme_cotation'] == 'francaise'].reset_index(drop=True)

# Replace the column values directly with lowercase values
df['difficulty'] = df['difficulty'].str.lower()

In [29]:
# Remove the final descent information from the descriptions 

motif_descente = re.compile(
    r"(?:\\n|\r?\n)[ \t]*(?:#{1,6}[ \t]*)?"
    r"(?:Descente|Pour la descente)\b",
    flags=re.IGNORECASE
)

def supprimer_descente(description):
    if pd.isna(description):
        return description

    texte = str(description)
    match = motif_descente.search(texte)

    if match:
        return texte[:match.start()].rstrip()

    return texte.strip()

df["description"] = df["description"].apply(supprimer_descente)


## 7. Creating a variable required for the analysis 

In [30]:
for i in range(len(df)):
    df.loc[i, 'log_nb_terms_description'] = np.log(len(df['description'][i].split()))

## 8. Saving the final dataset

In [31]:
df.to_csv("voies_par_longueur_annotation_cotationfr.csv", index=False, encoding="utf-8")